# SigAlg's `L2.metric` method

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2.metric` method in SigAlg computes the *$L^2$-distance* between two random variables in an $L^2$-space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2.metric).

## Mathematical definition

Let $X, Y \in L^2(\Omega, \mathcal{F}, P)$ be two square-integrable random variables on a probability space $(\Omega, \mathcal{F}, P)$. The *$L^2$-metric* (or *$L^2$-distance*) between $X$ and $Y$ is defined as

$$
d(X, Y) \stackrel{\text{def}}{=} \|X - Y\| = \sqrt{E\left[(X - Y)^2\right]} = \sqrt{\int_\Omega (X - Y)^2 \, dP}.
$$

The $L^2$-metric satisfies the following properties that make $(L^2(\Omega, \mathcal{F}, P), d)$ a *metric space*:

1. *Positivity*: For all $X, Y \in L^2$,
   $$
   d(X, Y) \geq 0,
   $$
   with equality if and only if $X = Y$ almost surely.

2. *Symmetry*: For all $X, Y \in L^2$,
   $$
   d(X, Y) = d(Y, X).
   $$

3. *Triangle inequality*: For all $X, Y, Z \in L^2$,
   $$
   d(X, Z) \leq d(X, Y) + d(Y, Z).
   $$

## API examples


### Basic distances

We begin by setting up a probability space and creating an $L^2$-space.

In [2]:
from sigalg.core import ProbabilityMeasure, RandomVariable, SampleSpace, SigmaAlgebra
from sigalg.l2 import L2

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

H = L2(sample_space=Omega, sig_alg=F, prob_measure=P)

print(H)

H = L2(Omega, F, P)

* Sample space 'Omega':
[0, 1, 2, 3]

* Sigma algebra 'F':
        atom ID
sample         
0             0
1             1
2             0
3             1

* Probability measure 'P':
        probability
sample             
0              0.10
1              0.15
2              0.45
3              0.30


Create two random variables and compute their $L^2$-distance.

In [11]:
X = (
    RandomVariable(domain=Omega, name="X")
    .from_dict(
        {
            0: 2,
            1: -1,
            2: 2,
            3: -1,
        }
    )
    .with_probability_measure(P)
)

Y = (
    RandomVariable(domain=Omega, name="Y")
    .from_dict(
        {
            0: 3,
            1: 5,
            2: 3,
            3: 5,
        }
    )
    .with_probability_measure(P)
)

distance_XY = H.metric(X, Y)
print(X)
print(Y)
print(f"d(X, Y) = {distance_XY:.6f}")

Random variable 'X':
        X
sample   
0       2
1      -1
2       2
3      -1
Random variable 'Y':
        Y
sample   
0       3
1       5
2       3
3       5
d(X, Y) = 4.092676


We can verify that the distance equals $\|X - Y\|$.

In [12]:
norm_diff = H.norm(X - Y)
print(f"d(X, Y) = {distance_XY:.6f}")
print(f"||X - Y|| = {norm_diff:.6f}")

d(X, Y) = 4.092676
||X - Y|| = 4.092676


### Symmetry

The metric is symmetric: $d(X, Y) = d(Y, X)$.

In [13]:
distance_YX = H.metric(Y, X)
print(f"d(X, Y) = {distance_XY:.6f}")
print(f"d(Y, X) = {distance_YX:.6f}")

d(X, Y) = 4.092676
d(Y, X) = 4.092676


### Triangle inequality

The metric satisfies the triangle inequality: $d(X, Z) \leq d(X, Y) + d(Y, Z)$.

In [15]:
Z = RandomVariable(domain=Omega, name="Z").from_dict(
    {
        0: 0,
        1: 2,
        2: 0,
        3: 2,
    }
).with_probability_measure(P)

distance_XZ = H.metric(X, Z)
distance_XY_plus_YZ = H.metric(X, Y) + H.metric(Y, Z)

print(f"d(X, Z) = {distance_XZ:.6f}")
print(f"d(X, Y) + d(Y, Z) = {distance_XY_plus_YZ:.6f}")

d(X, Z) = 2.500000
d(X, Y) + d(Y, Z) = 7.092676


### Distance between orthogonal random variables

For two orthogonal random variables $X$ and $Y$ (meaning $\langle X, Y \rangle = 0$), the distance satisfies the *Pythagorean theorem*:

$$
d(X, Y)^2 = \|X - Y\|^2 = \|X\|^2 + \|Y\|^2.
$$

Let's verify this with the orthonormal basis vectors.

In [16]:
basis = H.basis
basis_list = list(basis.values())

if len(basis_list) >= 2:
    phi_0 = basis_list[0]
    phi_1 = basis_list[1]

    # Verify orthogonality
    inner_prod = H.inner(phi_0, phi_1)
    print(f"⟨φ_0, φ_1⟩ = {inner_prod:.10f}")

    # Pythagorean theorem
    distance_squared = H.metric(phi_0, phi_1) ** 2
    sum_of_squares = H.norm(phi_0) ** 2 + H.norm(phi_1) ** 2

    print(f"d(φ_0, φ_1)² = {distance_squared:.6f}")
    print(f"‖φ_0‖² + ‖φ_1‖² = {sum_of_squares:.6f}")
    print(
        f"Pythagorean theorem holds: {abs(distance_squared - sum_of_squares) < 1e-10}"
    )

⟨φ_0, φ_1⟩ = 0.0000000000
d(φ_0, φ_1)² = 2.000000
‖φ_0‖² + ‖φ_1‖² = 2.000000
Pythagorean theorem holds: True


### Measuring similarity

The $L^2$ distance can be used to measure how similar two random variables are. Smaller distances indicate greater similarity.

In [8]:
# Create several random variables
X1 = RandomVariable(domain=Omega).from_dict({0: 1, 1: 2, 2: 1, 3: 2})
X2 = RandomVariable(domain=Omega).from_dict({0: 1.1, 1: 2.1, 2: 1.1, 3: 2.1})  # Close to X1
X3 = RandomVariable(domain=Omega).from_dict({0: 5, 1: -3, 2: 5, 3: -3})  # Far from X1

print(f"d(X1, X2) = {H.metric(X1, X2):.6f}  (similar)")
print(f"d(X1, X3) = {H.metric(X1, X3):.6f}  (dissimilar)")

d(X1, X2) = 0.100000  (similar)
d(X1, X3) = 4.477723  (dissimilar)


### Connection to mean squared error

The square of the $L^2$ distance is the **mean squared error (MSE)**:

$$
\text{MSE}(X, Y) = E\left[(X - Y)^2\right] = d(X, Y)^2.
$$

This is a fundamental quantity in regression and prediction problems.

In [9]:
import numpy as np

# Compute MSE manually
diff_squared = (X - Y)**2
P.prob_measure = P  # Ensure we can integrate
mse_manual = sum(diff_squared.data[i] * P.data[i] for i in range(4))

# From L2 distance
mse_from_distance = H.metric(X, Y)**2

print(f"MSE(X, Y) computed manually: {mse_manual:.6f}")
print(f"MSE(X, Y) from d(X,Y)²: {mse_from_distance:.6f}")
print(f"Match: {abs(mse_manual - mse_from_distance) < 1e-10}")

MSE(X, Y) computed manually: 16.750000
MSE(X, Y) from d(X,Y)²: 16.750000
Match: True


### Finding the closest random variable in a subspace

The orthogonal projection minimizes the $L^2$ distance. Given a random variable $X$ and a subspace $V \subset L^2$, the projection $\text{proj}_V(X)$ is the unique element of $V$ that minimizes $d(X, Y)$ over all $Y \in V$.

For details on projections, see the [`proj` notebook](proj.ipynb).

In [10]:
# Create a random variable not in the subspace spanned by basis vectors
basis_list = list(H.basis.values())
phi_0 = basis_list[0]

# Project X onto the 1-dimensional subspace spanned by phi_0
X_proj, coeffs, dim = H.proj(X, [phi_0])

# The projection minimizes distance to X
distance_to_proj = H.metric(X, X_proj)

# Try another element in the subspace
other_element = 2.5 * phi_0  # Different scalar multiple
distance_to_other = H.metric(X, other_element)

print(f"Distance from X to its projection: {distance_to_proj:.6f}")
print(f"Distance from X to another element: {distance_to_other:.6f}")
print(f"Projection minimizes distance: {distance_to_proj <= distance_to_other + 1e-10}")

Distance from X to its projection: 0.670820
Distance from X to another element: 1.218114
Projection minimizes distance: True
